# release16_failed_fold_daily_path_diagnostic_v1

Objective: inspect the remaining release16 checkpoint failures after the metric-basis repair, using the same `official_daily_path_rebased_eval_window` values that now feed the model-retention frontier and dashboard.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison: `release16_cb99_rebound_checkpoint_screen_v1`.
- Reference baseline: `soxlbacktest_qlib_walkforward_baseline_v1` with `legacy_default`, `risk_cap_35_15_05`, and `wf_pass_ytd18_15_15_02` as buy/sell reference rows.
- Role: `diagnose`.
- Target windows: failed release16 folds `wf_2020` and `wf_2023_2024`, with `wf_2022` as a passing guardrail.
- Changed layer: `model_selection`, `target_design`, `retention_gate_feasibility`.
- External reference: qlib-aligned SOXLBacktest buy/sell baseline and release16 repaired target contract.

Hypothesis: after fixing metric basis, release16 is not blocked by the old warmup benchmark artifact. The remaining failures are low-participation return gaps: `wf_2020` misses the post-crash/late-2020 rally while staying under MaxDD, and `wf_2023_2024` is too flat in a mild positive 2024 test window. If this is true, more PPO from the same BC target should not be promoted; the next mechanism should be a training-safe participation target/selection revision and a rerun of the three-fold checkpoint screen.

Acceptance for this diagnostic: produce runnable sidecars that show where the daily-path relative gaps come from, whether SOXLBacktest references make the gate feasible, whether the target contract teaches enough high-position states, and what the next non-promotion action should be.


In [1]:

from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    raise RuntimeError(f'could not locate repo root from {Path.cwd()}')

EXPERIMENT = 'release16_failed_fold_daily_path_diagnostic_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_EXPERIMENT = 'release16_cb99_rebound_checkpoint_screen_v1'
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / SOURCE_EXPERIMENT
TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'rebound_state_factor_target_repair_screen_v1'
SOXLBT_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_walkforward_baseline_v1'
FRONTIER_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'model_retention_frontier_v1'

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
FAILED_FOLDS = ['wf_2020', 'wf_2023_2024']
GUARD_FOLDS = ['wf_2022']
REFERENCE_CANDIDATES = ['legacy_default', 'risk_cap_35_15_05', 'wf_pass_ytd18_15_15_02']
FOLD_TARGET_MAP = {
    'wf_2020': 'wf_2020_train',
    'wf_2022': 'wf_2022_train',
    'wf_2023_2024': 'wf_2023_2024_train',
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def pct_return(values: pd.Series) -> float:
    clean = pd.to_numeric(values, errors='coerce').dropna()
    if len(clean) < 2:
        return 0.0
    start = float(clean.iloc[0])
    end = float(clean.iloc[-1])
    if not math.isfinite(start) or start == 0.0:
        return float('nan')
    return (end / start - 1.0) * 100.0


def max_dd_pct(values: pd.Series) -> float:
    clean = pd.to_numeric(values, errors='coerce').dropna()
    if clean.empty:
        return 0.0
    dd = clean / clean.cummax() - 1.0
    return float(-dd.min() * 100.0)


def normalize_daily(frame: pd.DataFrame, *, source: str, candidate: str, fold: str) -> pd.DataFrame:
    out = frame.copy()
    date_col = 'datetime' if 'datetime' in out.columns else 'date'
    out['date'] = pd.to_datetime(out[date_col]).dt.strftime('%Y-%m-%d')
    out['source'] = source
    out['candidate'] = candidate
    out['fold'] = fold
    if 'close' not in out.columns and 'price' in out.columns:
        out['close'] = out['price']
    if 'price' not in out.columns and 'close' in out.columns:
        out['price'] = out['close']
    return out.sort_values('date').reset_index(drop=True)


def path_metrics(frame: pd.DataFrame) -> dict[str, float | int | str]:
    if frame.empty:
        return {'rows': 0, 'start': '', 'end': '', 'return_pct': 0.0, 'buyhold_return_pct': 0.0, 'relative_return_pp': 0.0, 'max_dd_pct': 0.0, 'avg_position': 0.0}
    return {
        'rows': int(len(frame)),
        'start': str(frame['date'].iloc[0]),
        'end': str(frame['date'].iloc[-1]),
        'return_pct': pct_return(frame['strategy_nav']),
        'buyhold_return_pct': pct_return(frame['buyhold_nav']),
        'relative_return_pp': pct_return(frame['strategy_nav']) - pct_return(frame['buyhold_nav']),
        'max_dd_pct': max_dd_pct(frame['strategy_nav']),
        'avg_position': float(pd.to_numeric(frame['position'], errors='coerce').fillna(0.0).mean()),
    }


def stage_slices(fold: str, frame: pd.DataFrame) -> list[tuple[str, str, str]]:
    start = str(frame['date'].iloc[0])
    end = str(frame['date'].iloc[-1])
    if fold == 'wf_2020':
        return [
            ('full_window', start, end),
            ('pre_crash', '2020-01-02', '2020-02-21'),
            ('crash', '2020-02-24', '2020-03-23'),
            ('rebound', '2020-03-24', end),
            ('late_2020_rally', '2020-10-01', end),
        ]
    if fold == 'wf_2023_2024':
        return [
            ('full_window', start, end),
            ('q1', '2024-01-02', '2024-03-31'),
            ('q2', '2024-04-01', '2024-06-30'),
            ('q3', '2024-07-01', '2024-09-30'),
            ('q4', '2024-10-01', end),
        ]
    return [('full_window', start, end)]


def slice_between(frame: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    return frame[(frame['date'] >= start) & (frame['date'] <= end)].copy()


def position_bucket(pos: float) -> str:
    if pos < 0.10:
        return '00_cash_lt10'
    if pos < 0.30:
        return '10_low_10_30'
    if pos < 0.50:
        return '30_mid_30_50'
    if pos < 0.70:
        return '50_high_50_70'
    return '70_full_ge70'

print('card', CARD.relative_to(ROOT))


card runtime/model_selection/card/release16_failed_fold_daily_path_diagnostic_v1


In [2]:

required_paths = {
    'source_summary': SOURCE_CARD / 'summary.json',
    'source_candidates': SOURCE_CARD / 'checkpoint_candidate_metrics.csv',
    'source_selection': SOURCE_CARD / 'checkpoint_selection_by_fold.csv',
    'metric_basis_reconciliation': SOURCE_CARD / 'metric_basis_reconciliation.csv',
    'target_contract': TARGET_CARD / 'combined_target_contract.csv',
    'soxlbt_metrics': SOXLBT_CARD / 'candidate_metrics.csv',
    'soxlbt_daily': SOXLBT_CARD / 'candidate_daily.csv',
    'soxlbt_trades': SOXLBT_CARD / 'candidate_trades.csv',
    'frontier_summary': FRONTIER_CARD / 'summary.json',
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'missing required inputs: {missing}')

source_summary = json.loads(required_paths['source_summary'].read_text(encoding='utf-8'))
frontier_summary = json.loads(required_paths['frontier_summary'].read_text(encoding='utf-8'))
candidate_metrics = pd.read_csv(required_paths['source_candidates'])
selection_by_fold = pd.read_csv(required_paths['source_selection'])
metric_audit = pd.read_csv(required_paths['metric_basis_reconciliation'])
targets = pd.read_csv(required_paths['target_contract'])
targets['datetime'] = pd.to_datetime(targets['datetime']).dt.normalize()
soxlbt_metrics = pd.read_csv(required_paths['soxlbt_metrics'])
soxlbt_daily_raw = pd.read_csv(required_paths['soxlbt_daily'])
soxlbt_trades = pd.read_csv(required_paths['soxlbt_trades'])

source_selection = selection_by_fold.set_index('fold').to_dict('index')
print('source decision', source_summary.get('decision'))
print('failed_folds', source_summary.get('failed_folds'))
print('metric_basis', source_summary.get('metric_basis'))
print('frontier recommended', frontier_summary['summary'].get('recommended_next_source'), frontier_summary['summary'].get('recommended_primary_blocker'))
display(selection_by_fold)


source decision release16_cb99_checkpoint_metric_screen_partial_pass_no_promotion
failed_folds ['wf_2020', 'wf_2023_2024']
metric_basis official_daily_path_rebased_eval_window
frontier recommended release16_cb99_rebound_checkpoint_screen_v1 walkforward_relative_return_gap


,fold,status,selected_candidate,selected_kind,selected_iter,selected_target_gate,selected_drift_gate,selected_candidate_gate,selected_loss,selected_relative_return,...,candidate_count,reference_candidate,reference_return_pct,reference_relative_return_pp,reference_max_dd_pct,reference_avg_position,reference_trade_count,selected_vs_reference_return_delta_pct,selected_vs_reference_relative_delta_pp,selected_vs_reference_max_dd_delta_pct
0,wf_2020,evaluated,bc_initialization,bc_initialization,0,False,True,False,1.501076,-15.032278,...,4,wf_pass_ytd18_15_15_02,29.156603,-30.618705,12.785207,0.171011,2,13.904689,15.586427,2.176432
1,wf_2022,evaluated,bc_initialization,bc_initialization,0,True,True,True,0.000000,71.714980,...,4,wf_pass_ytd18_15_15_02,-16.044551,70.543521,18.514585,0.102318,6,1.171459,1.171459,-2.897542
2,wf_2023_2024,evaluated,bc_initialization,bc_initialization,0,False,True,False,0.843068,4.707959,...,4,wf_pass_ytd18_15_15_02,4.505779,7.109208,1.061804,0.013221,5,-0.154455,-2.401249,4.556762


In [3]:

# Load selected official daily paths and SOXLBacktest references for failed folds.
daily_frames: list[pd.DataFrame] = []
selected_daily_by_fold: dict[str, pd.DataFrame] = {}

for fold in FAILED_FOLDS:
    selected = source_selection[fold]
    selected_path = ROOT / str(selected['selected_daily_path'])
    selected_frame = normalize_daily(pd.read_csv(selected_path), source='release16_selected', candidate=str(selected['selected_candidate']), fold=fold)
    selected_daily_by_fold[fold] = selected_frame
    daily_frames.append(selected_frame)

    fold_candidates = candidate_metrics[candidate_metrics['fold'].eq(fold)].copy()
    for _, row in fold_candidates.iterrows():
        if row['candidate'] == selected['selected_candidate']:
            continue
        path = ROOT / str(row['daily_path'])
        if not path.exists():
            continue
        daily_frames.append(normalize_daily(pd.read_csv(path), source='release16_candidate', candidate=str(row['candidate']), fold=fold))

    for candidate in REFERENCE_CANDIDATES:
        ref = soxlbt_daily_raw[(soxlbt_daily_raw['candidate'].eq(candidate)) & (soxlbt_daily_raw['window'].eq(fold))].copy()
        if ref.empty:
            continue
        daily_frames.append(normalize_daily(ref, source='soxlbacktest_reference', candidate=candidate, fold=fold))

all_daily = pd.concat(daily_frames, ignore_index=True)
all_daily.to_csv(CARD / 'aligned_daily_paths.csv', index=False)

path_metric_rows = []
for (fold, source, candidate), group in all_daily.groupby(['fold', 'source', 'candidate'], sort=True):
    row = {'fold': fold, 'source': source, 'candidate': candidate, **path_metrics(group)}
    row['target_return_pct_for_plus30_relative'] = float(row['buyhold_return_pct']) + TARGET_RELATIVE_RETURN_PP
    row['return_gap_to_plus30_relative_pp'] = max(0.0, row['target_return_pct_for_plus30_relative'] - row['return_pct'])
    row['maxdd_gap_to_target_pp'] = max(0.0, row['max_dd_pct'] - TARGET_MAX_DD_PCT)
    row['gate_pass'] = bool(row['relative_return_pp'] >= TARGET_RELATIVE_RETURN_PP and row['max_dd_pct'] < TARGET_MAX_DD_PCT)
    path_metric_rows.append(row)
path_metric_frame = pd.DataFrame(path_metric_rows).sort_values(['fold', 'source', 'candidate']).reset_index(drop=True)
path_metric_frame.to_csv(CARD / 'daily_path_metric_reconciliation.csv', index=False)
display(path_metric_frame)


,fold,source,candidate,rows,start,end,return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_position,target_return_pct_for_plus30_relative,return_gap_to_plus30_relative_pp,maxdd_gap_to_target_pp,gate_pass
0,wf_2020,release16_candidate,ppo_retained_iter_0_bc_equivalent,252,2020-01-02,2020-12-30,43.061292,58.093571,-15.032278,14.961639,0.224446,88.093571,45.032278,0.000000,False
1,wf_2020,release16_candidate,ppo_retained_iter_4,252,2020-01-02,2020-12-30,54.701777,58.093571,-3.391793,20.939975,0.309147,88.093571,33.391793,0.939975,False
2,wf_2020,release16_candidate,ppo_selected_best,252,2020-01-02,2020-12-30,54.701777,58.093571,-3.391793,20.939975,0.309147,88.093571,33.391793,0.939975,False
3,wf_2020,release16_selected,bc_initialization,252,2020-01-02,2020-12-30,43.061292,58.093571,-15.032278,14.961639,0.224446,88.093571,45.032278,0.000000,False
4,wf_2020,soxlbacktest_reference,legacy_default,253,2020-01-02,2020-12-31,220.777205,59.775308,161.001897,54.275144,0.791023,89.775308,0.000000,34.275144,False
5,wf_2020,soxlbacktest_reference,risk_cap_35_15_05,253,2020-01-02,2020-12-31,69.604552,59.775308,9.829243,17.942350,0.310320,89.775308,20.170757,0.000000,False
6,wf_2020,soxlbacktest_reference,wf_pass_ytd18_15_15_02,253,2020-01-02,2020-12-31,29.156603,59.775308,-30.618705,12.785207,0.171011,89.775308,60.618705,0.000000,False
7,wf_2023_2024,release16_candidate,ppo_retained_iter_0_bc_equivalent,251,2024-01-02,2024-12-30,4.351324,-0.356635,4.707959,5.618566,0.085560,29.643365,25.292041,0.000000,False
8,wf_2023_2024,release16_candidate,ppo_retained_iter_4,251,2024-01-02,2024-12-30,2.240583,-0.356635,2.597218,5.503928,0.080085,29.643365,27.402782,0.000000,False
9,wf_2023_2024,release16_candidate,ppo_selected_best,251,2024-01-02,2024-12-30,2.240583,-0.356635,2.597218,5.503928,0.080085,29.643365,27.402782,0.000000,False


In [4]:

# Stage and monthly attribution: where selected release16 earns or misses return.
stage_rows = []
for (fold, source, candidate), group in all_daily.groupby(['fold', 'source', 'candidate'], sort=True):
    for stage, start, end in stage_slices(fold, group):
        window = slice_between(group, start, end)
        if window.empty:
            continue
        row = {'fold': fold, 'source': source, 'candidate': candidate, 'stage': stage, 'stage_start': start, 'stage_end': end, **path_metrics(window)}
        stage_rows.append(row)
stage_metrics = pd.DataFrame(stage_rows)
stage_metrics.to_csv(CARD / 'stage_path_metrics.csv', index=False)

monthly_rows = []
for (fold, source, candidate), group in all_daily.groupby(['fold', 'source', 'candidate'], sort=True):
    g = group.copy()
    g['month'] = pd.to_datetime(g['date']).dt.to_period('M').astype(str)
    for month, month_group in g.groupby('month'):
        row = {'fold': fold, 'source': source, 'candidate': candidate, 'month': month, **path_metrics(month_group)}
        monthly_rows.append(row)
monthly_metrics = pd.DataFrame(monthly_rows)
monthly_metrics.to_csv(CARD / 'monthly_path_metrics.csv', index=False)

selected_monthly = monthly_metrics[monthly_metrics['source'].eq('release16_selected')].copy()
ref_monthly = monthly_metrics[monthly_metrics['source'].eq('soxlbacktest_reference')].copy()
ref_pivot = ref_monthly.pivot_table(index=['fold', 'month'], columns='candidate', values=['return_pct', 'avg_position'], aggfunc='first')
ref_pivot.columns = [f'{metric}__{candidate}' for metric, candidate in ref_pivot.columns]
monthly_gap = selected_monthly.merge(ref_pivot.reset_index(), on=['fold', 'month'], how='left')
monthly_gap['selected_price_return_pct'] = monthly_gap['buyhold_return_pct']
for ref_candidate in REFERENCE_CANDIDATES:
    ret_col = f'return_pct__{ref_candidate}'
    pos_col = f'avg_position__{ref_candidate}'
    if ret_col in monthly_gap.columns:
        monthly_gap[f'return_delta_vs_{ref_candidate}_pp'] = monthly_gap['return_pct'] - monthly_gap[ret_col]
    if pos_col in monthly_gap.columns:
        monthly_gap[f'position_delta_vs_{ref_candidate}'] = monthly_gap['avg_position'] - monthly_gap[pos_col]
monthly_gap.to_csv(CARD / 'monthly_selected_vs_reference_gap.csv', index=False)

display(stage_metrics[(stage_metrics['source'].isin(['release16_selected','soxlbacktest_reference'])) & (stage_metrics['candidate'].isin(['bc_initialization','legacy_default','risk_cap_35_15_05','wf_pass_ytd18_15_15_02']))])
display(monthly_gap)


,fold,source,candidate,stage,stage_start,stage_end,rows,start,end,return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_position
15,wf_2020,release16_selected,bc_initialization,full_window,2020-01-02,2020-12-30,252,2020-01-02,2020-12-30,43.061292,58.093571,-15.032278,14.961639,0.224446
16,wf_2020,release16_selected,bc_initialization,pre_crash,2020-01-02,2020-02-21,35,2020-01-02,2020-02-21,0.000000,-2.914782,2.914782,-0.000000,0.160842
17,wf_2020,release16_selected,bc_initialization,crash,2020-02-24,2020-03-23,21,2020-02-24,2020-03-23,0.000000,-71.235752,71.235752,-0.000000,0.250112
18,wf_2020,release16_selected,bc_initialization,rebound,2020-03-24,2020-12-30,196,2020-03-24,2020-12-30,43.061292,392.393895,-349.332603,14.961639,0.233053
19,wf_2020,release16_selected,bc_initialization,late_2020_rally,2020-10-01,2020-12-30,63,2020-10-01,2020-12-30,19.982293,71.644785,-51.662493,9.149717,0.097279
20,wf_2020,soxlbacktest_reference,legacy_default,full_window,2020-01-02,2020-12-31,253,2020-01-02,2020-12-31,220.777205,59.775308,161.001897,54.275144,0.791023
21,wf_2020,soxlbacktest_reference,legacy_default,pre_crash,2020-01-02,2020-02-21,35,2020-01-02,2020-02-21,0.000000,-2.914782,2.914782,-0.000000,0.000000
22,wf_2020,soxlbacktest_reference,legacy_default,crash,2020-02-24,2020-03-23,21,2020-02-24,2020-03-23,-47.310881,-71.235752,23.924871,54.275144,0.532648
23,wf_2020,soxlbacktest_reference,legacy_default,rebound,2020-03-24,2020-12-31,197,2020-03-24,2020-12-31,366.212353,397.631789,-31.419436,30.060943,0.959103
24,wf_2020,soxlbacktest_reference,legacy_default,late_2020_rally,2020-10-01,2020-12-31,64,2020-10-01,2020-12-31,71.337144,73.470676,-2.133532,23.282001,0.975441


,fold,source,candidate,month,rows,start,end,return_pct,buyhold_return_pct,relative_return_pp,...,return_pct__legacy_default,return_pct__risk_cap_35_15_05,return_pct__wf_pass_ytd18_15_15_02,selected_price_return_pct,return_delta_vs_legacy_default_pp,position_delta_vs_legacy_default,return_delta_vs_risk_cap_35_15_05_pp,position_delta_vs_risk_cap_35_15_05,return_delta_vs_wf_pass_ytd18_15_15_02_pp,position_delta_vs_wf_pass_ytd18_15_15_02
0,wf_2020,release16_selected,bc_initialization,2020-01,21,2020-01-02,2020-01-31,0.000000,-16.560487,16.560487,...,0.000000,0.000000,0.000000,-16.560487,0.000000,0.159014,0.000000,0.159014,0.000000,0.159014
1,wf_2020,release16_selected,bc_initialization,2020-02,19,2020-02-03,2020-02-28,0.000000,-18.943494,18.943494,...,2.176667,1.088333,1.088333,-18.943494,-2.176667,0.151557,-1.088333,0.167646,-1.088333,0.167646
2,wf_2020,release16_selected,bc_initialization,2020-03,22,2020-03-02,2020-03-31,-0.023369,-57.949402,57.926033,...,-34.055269,-11.189014,-9.912397,-57.949402,34.031900,-0.433623,11.165644,0.118177,9.889028,0.149886
3,wf_2020,release16_selected,bc_initialization,2020-04,21,2020-04-01,2020-04-30,9.924802,64.979221,-55.054419,...,59.003468,12.374187,5.591467,64.979221,-49.078666,-0.671669,-2.449385,0.026355,4.333335,0.147217
4,wf_2020,release16_selected,bc_initialization,2020-05,20,2020-05-01,2020-05-29,6.431971,38.712863,-32.280892,...,36.099625,9.585744,4.505907,38.712863,-29.667655,-0.730852,-3.153773,-0.070952,1.926064,0.075648
5,wf_2020,release16_selected,bc_initialization,2020-06,22,2020-06-01,2020-06-30,6.426099,23.659654,-17.233555,...,22.472189,7.352173,3.616824,23.659654,-16.046090,-0.620345,-0.926074,0.012682,2.809275,0.173164
6,wf_2020,release16_selected,bc_initialization,2020-07,22,2020-07-01,2020-07-31,6.419677,25.601641,-19.181964,...,24.508648,8.914382,4.509825,25.601641,-18.088971,-0.654052,-2.494705,-0.063206,1.909852,0.116294
7,wf_2020,release16_selected,bc_initialization,2020-08,21,2020-08-03,2020-08-31,2.163666,11.296308,-9.132642,...,10.930024,4.693747,2.502274,11.296308,-8.766357,-0.630413,-2.530081,-0.069842,-0.338608,0.121130
8,wf_2020,release16_selected,bc_initialization,2020-09,21,2020-09-01,2020-09-30,-6.192436,-10.260865,4.068429,...,-9.976482,-4.671412,-2.572043,-10.260865,3.784046,-0.664318,-1.521024,-0.114318,-3.620393,0.080349
9,wf_2020,release16_selected,bc_initialization,2020-10,22,2020-10-01,2020-10-30,-1.370920,-7.422562,6.051642,...,-7.207017,-3.291032,-1.794425,-7.422562,5.836097,-0.874066,1.920112,-0.351684,0.423505,-0.149321


In [5]:

# Daily missed-up attribution: selected low exposure on large positive SOXL days.
missed_rows = []
for fold in FAILED_FOLDS:
    selected = selected_daily_by_fold[fold].copy()
    selected['soxl_daily_return'] = pd.to_numeric(selected['price'], errors='coerce').pct_change().fillna(0.0)
    selected['selected_prev_position'] = pd.to_numeric(selected['position'], errors='coerce').shift(1).fillna(0.0)
    ref = all_daily[(all_daily['fold'].eq(fold)) & (all_daily['source'].eq('soxlbacktest_reference')) & (all_daily['candidate'].eq('risk_cap_35_15_05'))][['date','position']].copy()
    ref = ref.rename(columns={'position': 'risk_cap_position'})
    legacy = all_daily[(all_daily['fold'].eq(fold)) & (all_daily['source'].eq('soxlbacktest_reference')) & (all_daily['candidate'].eq('legacy_default'))][['date','position']].copy()
    legacy = legacy.rename(columns={'position': 'legacy_position'})
    joined = selected.merge(ref, on='date', how='left').merge(legacy, on='date', how='left')
    joined['risk_cap_prev_position'] = pd.to_numeric(joined['risk_cap_position'], errors='coerce').shift(1).fillna(0.0)
    joined['legacy_prev_position'] = pd.to_numeric(joined['legacy_position'], errors='coerce').shift(1).fillna(0.0)
    joined['selected_vs_risk_cap_prev_pos_gap'] = joined['risk_cap_prev_position'] - joined['selected_prev_position']
    joined['selected_vs_legacy_prev_pos_gap'] = joined['legacy_prev_position'] - joined['selected_prev_position']
    joined['incremental_return_if_risk_cap_pos'] = joined['soxl_daily_return'] * joined['selected_vs_risk_cap_prev_pos_gap'] * 100.0
    joined['incremental_return_if_legacy_pos'] = joined['soxl_daily_return'] * joined['selected_vs_legacy_prev_pos_gap'] * 100.0
    up = joined[(joined['soxl_daily_return'] > 0.03) & (joined['selected_prev_position'] < 0.30)].copy()
    up['fold'] = fold
    missed_rows.append(up)

missed_up_days = pd.concat(missed_rows, ignore_index=True) if missed_rows else pd.DataFrame()
missed_up_days = missed_up_days.sort_values(['fold', 'incremental_return_if_risk_cap_pos'], ascending=[True, False])
missed_up_days.to_csv(CARD / 'missed_up_days_low_position.csv', index=False)

missed_summary = missed_up_days.groupby('fold').agg(
    days=('date','count'),
    mean_selected_prev_position=('selected_prev_position','mean'),
    mean_risk_cap_prev_position=('risk_cap_prev_position','mean'),
    mean_legacy_prev_position=('legacy_prev_position','mean'),
    sum_soxl_up_day_return_pp=('soxl_daily_return', lambda s: float(s.sum()) * 100.0),
    mean_soxl_up_day_return_pp=('soxl_daily_return', lambda s: float(s.mean()) * 100.0),
    sum_incremental_if_risk_cap_pos_pp=('incremental_return_if_risk_cap_pos','sum'),
    sum_incremental_if_legacy_pos_pp=('incremental_return_if_legacy_pos','sum'),
).reset_index()
missed_summary.to_csv(CARD / 'missed_up_days_summary.csv', index=False)
display(missed_summary)
display(missed_up_days[['fold','date','soxl_daily_return','selected_prev_position','risk_cap_prev_position','legacy_prev_position','incremental_return_if_risk_cap_pos','incremental_return_if_legacy_pos']].head(30))


,fold,days,mean_selected_prev_position,mean_risk_cap_prev_position,mean_legacy_prev_position,sum_soxl_up_day_return_pp,mean_soxl_up_day_return_pp,sum_incremental_if_risk_cap_pos_pp,sum_incremental_if_legacy_pos_pp
0,wf_2020,74,0.159178,0.306404,0.781132,540.952480,7.310169,68.109071,333.249405
1,wf_2023_2024,85,0.079414,0.032135,0.313069,563.111812,6.624845,-25.449755,137.888861


,fold,date,soxl_daily_return,selected_prev_position,risk_cap_prev_position,legacy_prev_position,incremental_return_if_risk_cap_pos,incremental_return_if_legacy_pos
60,wf_2020,2020-11-05,0.131881,0.097279,0.4791,0.9748,5.035493,11.572838
62,wf_2020,2020-11-11,0.108964,0.097279,0.4791,0.9748,4.160479,9.561835
59,wf_2020,2020-11-04,0.104810,0.097279,0.4791,0.9748,4.001846,9.197256
70,wf_2020,2020-12-04,0.083239,0.097279,0.4934,0.9761,3.297255,7.315185
52,wf_2020,2020-10-05,0.097616,0.097279,0.4128,0.9672,3.079979,8.491795
57,wf_2020,2020-10-29,0.079624,0.097279,0.4791,0.9748,3.040200,6.987150
64,wf_2020,2020-11-16,0.073749,0.097279,0.4934,0.9761,2.921363,6.481243
48,wf_2020,2020-09-02,0.089707,0.105397,0.4225,0.9685,2.844644,7.742661
42,wf_2020,2020-07-27,0.094725,0.096924,0.3930,0.9645,2.804567,8.218084
25,wf_2020,2020-04-29,0.143482,0.095025,0.2781,0.9418,2.626804,12.149711


In [6]:

# Target contract distribution: does the repaired target teach high participation for failed folds?
target_rows = []
bucket_rows = []
feature_cols = ['mom_5','mom_21','drawdown_63','rv_21','vol_of_vol_21','soxl_soxx_ratio_widen_21','soxl_soxx_ratio_z_252','soxx_mom_21']
for fold in FAILED_FOLDS + GUARD_FOLDS:
    target_fold = FOLD_TARGET_MAP[fold]
    group = targets[targets['fold'].astype(str).eq(target_fold)].copy()
    if group.empty:
        continue
    pos = pd.to_numeric(group['risk_target_position'], errors='coerce').fillna(0.0)
    row = {
        'fold': fold,
        'target_fold': target_fold,
        'rows': int(len(group)),
        'start': str(group['datetime'].min().date()),
        'end': str(group['datetime'].max().date()),
        'mean_target_position': float(pos.mean()),
        'median_target_position': float(pos.median()),
        'max_target_position': float(pos.max()),
        'share_ge_30pct': float((pos >= 0.30).mean()),
        'share_ge_50pct': float((pos >= 0.50).mean()),
        'share_ge_70pct': float((pos >= 0.70).mean()),
        'rows_ge_70pct': int((pos >= 0.70).sum()),
        'release_rows': int(group.get('release_gate_state', pd.Series(False, index=group.index)).fillna(False).astype(bool).sum()),
        'rebound_rows': int(group.get('is_rebound_cluster', pd.Series(False, index=group.index)).fillna(False).astype(bool).sum()),
        'downtrend_rows': int(group.get('is_downtrend_state', pd.Series(False, index=group.index)).fillna(False).astype(bool).sum()),
        'drawdown_control_rows': int(group.get('drawdown_control_state', pd.Series(False, index=group.index)).fillna(False).astype(bool).sum()),
    }
    target_rows.append(row)
    tmp = pd.DataFrame({'bucket': [position_bucket(float(x)) for x in pos]})
    counts = tmp['bucket'].value_counts().rename_axis('bucket').reset_index(name='rows')
    counts['fold'] = fold
    counts['target_fold'] = target_fold
    counts['share'] = counts['rows'] / max(len(group), 1)
    bucket_rows.append(counts)

target_distribution = pd.DataFrame(target_rows)
target_buckets = pd.concat(bucket_rows, ignore_index=True) if bucket_rows else pd.DataFrame()
target_distribution.to_csv(CARD / 'target_distribution_by_fold.csv', index=False)
target_buckets.to_csv(CARD / 'target_position_buckets.csv', index=False)

# Compare failed-fold missed-up days to training target feature states when columns exist.
state_rows = []
for fold in FAILED_FOLDS:
    target_fold = FOLD_TARGET_MAP[fold]
    group = targets[targets['fold'].astype(str).eq(target_fold)].copy()
    high = group[pd.to_numeric(group['risk_target_position'], errors='coerce').fillna(0.0) >= 0.50]
    low = group[pd.to_numeric(group['risk_target_position'], errors='coerce').fillna(0.0) < 0.30]
    for feature in feature_cols:
        if feature not in group.columns:
            continue
        state_rows.append({
            'fold': fold,
            'feature': feature,
            'target_high_pos_mean': float(pd.to_numeric(high[feature], errors='coerce').mean()) if not high.empty else None,
            'target_low_pos_mean': float(pd.to_numeric(low[feature], errors='coerce').mean()) if not low.empty else None,
            'target_high_minus_low': (float(pd.to_numeric(high[feature], errors='coerce').mean()) - float(pd.to_numeric(low[feature], errors='coerce').mean())) if not high.empty and not low.empty else None,
            'high_rows': int(len(high)),
            'low_rows': int(len(low)),
        })
target_feature_contrast = pd.DataFrame(state_rows)
target_feature_contrast.to_csv(CARD / 'target_feature_contrast.csv', index=False)

display(target_distribution)
display(target_buckets.sort_values(['fold','bucket']))
display(target_feature_contrast)


,fold,target_fold,rows,start,end,mean_target_position,median_target_position,max_target_position,share_ge_30pct,share_ge_50pct,share_ge_70pct,rows_ge_70pct,release_rows,rebound_rows,downtrend_rows,drawdown_control_rows
0,wf_2020,wf_2020_train,2470,2010-03-11,2019-12-31,0.312066,0.250000,0.9155,0.383806,0.246154,0.008502,21,42,42,781,781
1,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,0.154589,0.105063,1.0000,0.100529,0.044974,0.029101,22,11,14,478,0
2,wf_2022,wf_2022_train,505,2020-01-02,2021-12-31,0.198093,0.146277,1.0000,0.146535,0.067327,0.043564,22,9,13,234,0


,bucket,rows,fold,target_fold,share
3,00_cash_lt10,309,wf_2020,wf_2020_train,0.125101
0,10_low_10_30,1213,wf_2020,wf_2020_train,0.491093
2,30_mid_30_50,340,wf_2020,wf_2020_train,0.137652
1,50_high_50_70,587,wf_2020,wf_2020_train,0.237652
4,70_full_ge70,21,wf_2020,wf_2020_train,0.008502
11,00_cash_lt10,168,wf_2022,wf_2022_train,0.332673
10,10_low_10_30,263,wf_2022,wf_2022_train,0.520792
12,30_mid_30_50,40,wf_2022,wf_2022_train,0.079208
14,50_high_50_70,12,wf_2022,wf_2022_train,0.023762
13,70_full_ge70,22,wf_2022,wf_2022_train,0.043564


,fold,feature,target_high_pos_mean,target_low_pos_mean,target_high_minus_low,high_rows,low_rows
0,wf_2020,mom_5,NaN,NaN,NaN,608,1522
1,wf_2020,mom_21,NaN,NaN,NaN,608,1522
2,wf_2020,drawdown_63,NaN,NaN,NaN,608,1522
3,wf_2020,rv_21,NaN,NaN,NaN,608,1522
4,wf_2020,vol_of_vol_21,NaN,NaN,NaN,608,1522
5,wf_2020,soxl_soxx_ratio_widen_21,NaN,NaN,NaN,608,1522
6,wf_2020,soxl_soxx_ratio_z_252,NaN,NaN,NaN,608,1522
7,wf_2020,soxx_mom_21,NaN,NaN,NaN,608,1522
8,wf_2023_2024,mom_5,0.094694,0.002932,0.091763,34,680
9,wf_2023_2024,mom_21,0.335703,0.002999,0.332704,34,680


In [7]:

# Candidate behavior decomposition: did PPO solve the failed folds?
source_candidates = candidate_metrics.copy()
selected = selection_by_fold[['fold','selected_candidate','selected_relative_return','selected_max_dd','selected_return','selected_avg_pos','selected_candidate_gate']].copy()

candidate_decomp_rows = []
for fold in FAILED_FOLDS + GUARD_FOLDS:
    fold_rows = source_candidates[source_candidates['fold'].eq(fold)].copy()
    if fold_rows.empty:
        continue
    bc = fold_rows[fold_rows['candidate'].eq('bc_initialization')].iloc[0]
    best_rel = fold_rows.sort_values('test_relative_return', ascending=False).iloc[0]
    best_loss = fold_rows.sort_values('selection_loss', ascending=True).iloc[0]
    candidate_decomp_rows.append({
        'fold': fold,
        'bc_relative_return_pp': float(bc['test_relative_return']),
        'bc_return_pct': float(bc['test_return']),
        'bc_max_dd_pct': float(bc['test_max_dd']),
        'bc_avg_pos': float(bc['test_avg_pos']),
        'best_relative_candidate': str(best_rel['candidate']),
        'best_relative_return_pp': float(best_rel['test_relative_return']),
        'best_relative_return_lift_vs_bc_pp': float(best_rel['test_relative_return'] - bc['test_relative_return']),
        'best_relative_max_dd_pct': float(best_rel['test_max_dd']),
        'best_relative_max_dd_delta_vs_bc_pp': float(best_rel['test_max_dd'] - bc['test_max_dd']),
        'best_loss_candidate': str(best_loss['candidate']),
        'best_loss_relative_return_pp': float(best_loss['test_relative_return']),
        'best_loss_max_dd_pct': float(best_loss['test_max_dd']),
        'any_candidate_gate': bool(fold_rows['candidate_gate'].astype(bool).any()),
        'candidate_gate_count': int(fold_rows['candidate_gate'].astype(bool).sum()),
        'target_gate_count': int(fold_rows['target_gate'].astype(bool).sum()),
        'drift_gate_count': int(fold_rows['drift_gate'].astype(bool).sum()),
    })
candidate_decomposition = pd.DataFrame(candidate_decomp_rows)
candidate_decomposition.to_csv(CARD / 'checkpoint_failure_decomposition.csv', index=False)
display(candidate_decomposition)


,fold,bc_relative_return_pp,bc_return_pct,bc_max_dd_pct,bc_avg_pos,best_relative_candidate,best_relative_return_pp,best_relative_return_lift_vs_bc_pp,best_relative_max_dd_pct,best_relative_max_dd_delta_vs_bc_pp,best_loss_candidate,best_loss_relative_return_pp,best_loss_max_dd_pct,any_candidate_gate,candidate_gate_count,target_gate_count,drift_gate_count
0,wf_2020,-15.032278,43.061292,14.961639,0.224446,ppo_retained_iter_4,-3.391793,11.640485,20.939975,5.978336,bc_initialization,-15.032278,14.961639,False,0,0,2
1,wf_2023_2024,4.707959,4.351324,5.618566,0.085560,bc_initialization,4.707959,0.000000,5.618566,0.000000,bc_initialization,4.707959,5.618566,False,0,0,4
2,wf_2022,71.714980,-14.873092,15.617043,0.081384,bc_initialization,71.714980,0.000000,15.617043,0.000000,bc_initialization,71.714980,15.617043,True,2,4,2


In [8]:

# Feasibility reference matrix against the daily-path +30pp relative gate.
reference_rows = []
for _, row in path_metric_frame.iterrows():
    if row['source'] not in {'release16_selected', 'soxlbacktest_reference'}:
        continue
    reference_rows.append({
        'fold': row['fold'],
        'source': row['source'],
        'candidate': row['candidate'],
        'return_pct': row['return_pct'],
        'buyhold_return_pct': row['buyhold_return_pct'],
        'relative_return_pp': row['relative_return_pp'],
        'max_dd_pct': row['max_dd_pct'],
        'avg_position': row['avg_position'],
        'required_return_for_plus30_relative_pct': row['target_return_pct_for_plus30_relative'],
        'return_gap_to_plus30_relative_pp': row['return_gap_to_plus30_relative_pp'],
        'maxdd_gap_to_target_pp': row['maxdd_gap_to_target_pp'],
        'gate_pass': row['gate_pass'],
    })
reference_matrix = pd.DataFrame(reference_rows).sort_values(['fold','source','candidate']).reset_index(drop=True)
reference_matrix.to_csv(CARD / 'reference_feasibility_matrix.csv', index=False)

fold_feasibility = []
for fold, group in reference_matrix.groupby('fold'):
    selected_row = group[(group['source'].eq('release16_selected'))].iloc[0]
    refs = group[group['source'].eq('soxlbacktest_reference')]
    feasible_refs = refs[refs['gate_pass'].astype(bool)]
    best_return_ref = refs.sort_values('return_pct', ascending=False).iloc[0] if not refs.empty else None
    best_dd_ref = refs.sort_values('max_dd_pct', ascending=True).iloc[0] if not refs.empty else None
    fold_feasibility.append({
        'fold': fold,
        'selected_return_pct': float(selected_row['return_pct']),
        'selected_relative_return_pp': float(selected_row['relative_return_pp']),
        'selected_max_dd_pct': float(selected_row['max_dd_pct']),
        'required_return_for_plus30_relative_pct': float(selected_row['required_return_for_plus30_relative_pct']),
        'selected_return_gap_to_plus30_relative_pp': float(selected_row['return_gap_to_plus30_relative_pp']),
        'reference_gate_pass_count': int(len(feasible_refs)),
        'best_reference_by_return': None if best_return_ref is None else str(best_return_ref['candidate']),
        'best_reference_return_pct': None if best_return_ref is None else float(best_return_ref['return_pct']),
        'best_reference_max_dd_pct': None if best_return_ref is None else float(best_return_ref['max_dd_pct']),
        'lowest_dd_reference': None if best_dd_ref is None else str(best_dd_ref['candidate']),
        'lowest_dd_reference_return_pct': None if best_dd_ref is None else float(best_dd_ref['return_pct']),
        'lowest_dd_reference_max_dd_pct': None if best_dd_ref is None else float(best_dd_ref['max_dd_pct']),
    })
fold_feasibility = pd.DataFrame(fold_feasibility)
fold_feasibility.to_csv(CARD / 'fold_feasibility_summary.csv', index=False)
display(reference_matrix)
display(fold_feasibility)


,fold,source,candidate,return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_position,required_return_for_plus30_relative_pct,return_gap_to_plus30_relative_pp,maxdd_gap_to_target_pp,gate_pass
0,wf_2020,release16_selected,bc_initialization,43.061292,58.093571,-15.032278,14.961639,0.224446,88.093571,45.032278,0.000000,False
1,wf_2020,soxlbacktest_reference,legacy_default,220.777205,59.775308,161.001897,54.275144,0.791023,89.775308,0.000000,34.275144,False
2,wf_2020,soxlbacktest_reference,risk_cap_35_15_05,69.604552,59.775308,9.829243,17.942350,0.310320,89.775308,20.170757,0.000000,False
3,wf_2020,soxlbacktest_reference,wf_pass_ytd18_15_15_02,29.156603,59.775308,-30.618705,12.785207,0.171011,89.775308,60.618705,0.000000,False
4,wf_2023_2024,release16_selected,bc_initialization,4.351324,-0.356635,4.707959,5.618566,0.085560,29.643365,25.292041,0.000000,False
5,wf_2023_2024,soxlbacktest_reference,legacy_default,2.935118,-2.603429,5.538546,32.461810,0.310435,27.396571,24.461454,12.461810,False
6,wf_2023_2024,soxlbacktest_reference,risk_cap_35_15_05,4.407749,-2.603429,7.011178,2.628936,0.027903,27.396571,22.988822,0.000000,False
7,wf_2023_2024,soxlbacktest_reference,wf_pass_ytd18_15_15_02,4.505779,-2.603429,7.109208,1.061804,0.013221,27.396571,22.890792,0.000000,False


,fold,selected_return_pct,selected_relative_return_pp,selected_max_dd_pct,required_return_for_plus30_relative_pct,selected_return_gap_to_plus30_relative_pp,reference_gate_pass_count,best_reference_by_return,best_reference_return_pct,best_reference_max_dd_pct,lowest_dd_reference,lowest_dd_reference_return_pct,lowest_dd_reference_max_dd_pct
0,wf_2020,43.061292,-15.032278,14.961639,88.093571,45.032278,0,legacy_default,220.777205,54.275144,wf_pass_ytd18_15_15_02,29.156603,12.785207
1,wf_2023_2024,4.351324,4.707959,5.618566,29.643365,25.292041,0,wf_pass_ytd18_15_15_02,4.505779,1.061804,wf_pass_ytd18_15_15_02,4.505779,1.061804


In [9]:

# Synthesize decision, selection, and summary.
failed_rows = selection_by_fold[selection_by_fold['fold'].isin(FAILED_FOLDS)].copy()
failed_fold_readout = {
    row['fold']: {
        'selected_candidate': row['selected_candidate'],
        'selected_return_pct': float(row['selected_return']),
        'selected_relative_return_pp': float(row['selected_relative_return']),
        'selected_max_dd_pct': float(row['selected_max_dd']),
        'selected_avg_position': float(row['selected_avg_pos']),
        'relative_gap_to_plus30_pp': float(max(0.0, TARGET_RELATIVE_RETURN_PP - row['selected_relative_return'])),
        'maxdd_gap_to_20pct_pp': float(max(0.0, row['selected_max_dd'] - TARGET_MAX_DD_PCT)),
    }
    for _, row in failed_rows.iterrows()
}

wf2020_stage = stage_metrics[(stage_metrics['fold'].eq('wf_2020')) & (stage_metrics['source'].eq('release16_selected')) & (stage_metrics['stage'].eq('late_2020_rally'))]
wf2023_q4 = stage_metrics[(stage_metrics['fold'].eq('wf_2023_2024')) & (stage_metrics['source'].eq('release16_selected')) & (stage_metrics['stage'].eq('q4'))]
wf2020_late = wf2020_stage.iloc[0].to_dict() if not wf2020_stage.empty else {}
wf2023_q4_row = wf2023_q4.iloc[0].to_dict() if not wf2023_q4.empty else {}

missed_summary_by_fold = missed_summary.set_index('fold').to_dict('index') if not missed_summary.empty else {}
target_distribution_by_fold = target_distribution.set_index('fold').to_dict('index') if not target_distribution.empty else {}
candidate_decomp_by_fold = candidate_decomposition.set_index('fold').to_dict('index') if not candidate_decomposition.empty else {}
feasibility_by_fold = fold_feasibility.set_index('fold').to_dict('index') if not fold_feasibility.empty else {}

# Classification: target revision is needed when failures are return-only, PPO does not find a gated candidate, and target high-position coverage is sparse.
return_only_failures = all(v['relative_gap_to_plus30_pp'] > 0 and v['maxdd_gap_to_20pct_pp'] <= 0 for v in failed_fold_readout.values())
ppo_no_gate = all(not candidate_decomp_by_fold.get(fold, {}).get('any_candidate_gate', False) for fold in FAILED_FOLDS)
reference_gap_exists = any(feasibility_by_fold.get(fold, {}).get('reference_gate_pass_count', 0) == 0 for fold in FAILED_FOLDS)
high_position_sparse = {
    fold: float(target_distribution_by_fold.get(fold, {}).get('share_ge_50pct', 0.0) or 0.0) < 0.10
    for fold in FAILED_FOLDS
}

if return_only_failures and ppo_no_gate:
    decision = 'release16_failed_folds_require_participation_target_revision_no_promotion'
    next_mechanism = 'build_release16_failedfold_participation_target_overlay_then_rerun_checkpoint_screen'
    primary_failure = 'daily_path_return_gap_from_low_participation'
else:
    decision = 'release16_failed_folds_need_additional_selection_debug_no_promotion'
    next_mechanism = 'inspect_release16_candidate_selection_and_metric_basis_before_target_revision'
    primary_failure = 'mixed_failure_requires_more_diagnostics'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': SOURCE_EXPERIMENT,
    'source_card': SOURCE_EXPERIMENT,
    'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'changed_layer': ['model_selection', 'target_design', 'retention_gate_feasibility'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'metric_basis': 'official_daily_path_rebased_eval_window',
    'failed_folds': FAILED_FOLDS,
    'guard_folds': GUARD_FOLDS,
    'primary_failure': primary_failure,
    'return_only_failures': bool(return_only_failures),
    'ppo_no_candidate_gate_on_failed_folds': bool(ppo_no_gate),
    'reference_gate_gap_exists': bool(reference_gap_exists),
    'high_position_sparse_by_failed_fold': high_position_sparse,
    'failed_fold_readout': failed_fold_readout,
    'fold_feasibility': feasibility_by_fold,
    'target_distribution': target_distribution_by_fold,
    'missed_up_days_summary': missed_summary_by_fold,
    'candidate_decomposition': candidate_decomp_by_fold,
    'next_mechanism': next_mechanism,
    'next_training_allowed_after_target_revision': True,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'promotion_blocker': 'Diagnostic only. No actor is trained or retained; rerun the release16 three-fold checkpoint screen after a training-safe target/selection revision.',
    'sidecars': [
        'aligned_daily_paths.csv',
        'daily_path_metric_reconciliation.csv',
        'stage_path_metrics.csv',
        'monthly_path_metrics.csv',
        'monthly_selected_vs_reference_gap.csv',
        'missed_up_days_low_position.csv',
        'missed_up_days_summary.csv',
        'target_distribution_by_fold.csv',
        'target_position_buckets.csv',
        'target_feature_contrast.csv',
        'checkpoint_failure_decomposition.csv',
        'reference_feasibility_matrix.csv',
        'fold_feasibility_summary.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}

conclusion_lines = [
    f"Decision: `{decision}`.",
    f"Metric basis: `official_daily_path_rebased_eval_window`; direct comparison source `{SOURCE_EXPERIMENT}`.",
    f"Failed folds remain return-only: {return_only_failures}; PPO/checkpoint screen produced no gated candidate on failed folds: {ppo_no_gate}.",
]
for fold in FAILED_FOLDS:
    readout = failed_fold_readout[fold]
    feasibility = feasibility_by_fold.get(fold, {})
    missed = missed_summary_by_fold.get(fold, {})
    target = target_distribution_by_fold.get(fold, {})
    conclusion_lines.append(
        f"{fold}: selected `{readout['selected_candidate']}` return {readout['selected_return_pct']:.2f}%, rel {readout['selected_relative_return_pp']:.2f}pp, "
        f"MaxDD {readout['selected_max_dd_pct']:.2f}%, return gap to +30pp relative {readout['relative_gap_to_plus30_pp']:.2f}pp."
    )
    if feasibility:
        conclusion_lines.append(
            f"{fold}: SOXLBacktest reference gate-pass count {int(feasibility.get('reference_gate_pass_count', 0))}; best return reference `{feasibility.get('best_reference_by_return')}` "
            f"return {float(feasibility.get('best_reference_return_pct') or 0.0):.2f}% / MaxDD {float(feasibility.get('best_reference_max_dd_pct') or 0.0):.2f}%."
        )
    if missed:
        conclusion_lines.append(
            f"{fold}: low-position up-day screen found {int(missed.get('days', 0))} days; selected prev-position mean {float(missed.get('mean_selected_prev_position') or 0.0):.3f}, "
            f"risk-cap reference prev-position mean {float(missed.get('mean_risk_cap_prev_position') or 0.0):.3f}."
        )
    if target:
        conclusion_lines.append(
            f"{fold}: target share >=50% is {float(target.get('share_ge_50pct') or 0.0):.3%}; share >=70% is {float(target.get('share_ge_70pct') or 0.0):.3%}."
        )
conclusion_lines.append(f"Next action: `{next_mechanism}`; keep actor retention blocked until the revised target reruns all three walk-forward folds and YTD/dashboard gates.")

summary = {
    **selection,
    'conclusion': conclusion_lines,
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

conclusion_md = ['# Release16 Failed Fold Daily-Path Diagnostic', ''] + [f'- {line}' for line in conclusion_lines] + [
    '',
    'Sidecars are written in this card directory. This diagnostic does not promote or retain any actor.',
]
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_md).rstrip() + '\n', encoding='utf-8')
print(json.dumps(json_safe(summary), indent=2, allow_nan=False)[:6000])


{
  "experiment": "release16_failed_fold_daily_path_diagnostic_v1",
  "created_at": "2026-06-14T14:30:06.279213+00:00",
  "decision": "release16_failed_folds_require_participation_target_revision_no_promotion",
  "role": "diagnose",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "release16_cb99_rebound_checkpoint_screen_v1",
  "source_card": "release16_cb99_rebound_checkpoint_screen_v1",
  "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
  "changed_layer": [
    "model_selection",
    "target_design",
    "retention_gate_feasibility"
  ],
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 20.0,
  "metric_basis": "official_daily_path_rebased_eval_window",
  "failed_folds": [
    "wf_2020",
    "wf_2023_2024"
  ],
  "guard_folds": [
    "wf_2022"
  ],
  "primary_failure": "daily_path_return_gap_from_low_participation",
  "return_only_failures": true,
  "ppo_no_candidate_gate_on_failed_folds": true,
  "reference_

## Conclusion

This notebook records the current release16 failure after metric-basis repair. The source card is no longer blocked by the old raw warmup benchmark artifact; the remaining failures are return-only daily-path gaps under the `+30pp relative` and `<20% MaxDD` gate. The sidecars in this card should be used to build a training-safe failed-fold participation target overlay, then rerun the release16 three-fold checkpoint screen before any retained actor training or dashboard promotion.